# Phase 1: Public Online Data Collection (Model L - Nepali)

This notebook downloads and saves the **non-manual** (public/online) datasets for the Nepali language model.

### Data Source: ai4bharat/IndicCorpV2

We use the **IndicCorpV2** dataset from AI4Bharat, which provides massive monolingual corpora for Indian languages.
The Nepali split (`npi_Deva`) contains billions of sentences crawled from the web.

- **HuggingFace**: `ai4bharat/IndicCorpV2` (config: `indiccorp_v2`, split: `npi_Deva`)
- **Streaming**: Dataset is streamed to avoid downloading the full 15GB+ archive
- **Target**: 500M+ words

---

## Summary

| File | Docs (Lines) | Est. Words | Size | Source |
| :--- | :---: | :---: | :---: | :--- |
| `nepali_indiccorp.jsonl` | ~20.7M | ~701M | ~13 GB | ai4bharat/IndicCorpV2 (npi_Deva) |
| `nepali_wikipedia.jsonl` | ~32.9K | ~5.6M | ~99 MB | Wikimedia (via HF) |
| **TOTAL** | **~20.7M** | **~706M** | **~13.1 GB** | |

## 1. Setup

In [ ]:
import json
from pathlib import Path
from datasets import load_dataset
from tqdm.auto import tqdm

RAW_DOWNLOAD_DIR = Path("../datasets/raw/downloaded/")
RAW_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {RAW_DOWNLOAD_DIR.resolve()}")

## 2. Download IndicCorpV2 — Nepali (500M+ words)

Streams the `npi_Deva` split of `ai4bharat/IndicCorpV2` and saves ~20M+ documents to JSONL.
Each document is a short sentence/paragraph averaging ~17 words.

**Runtime**: ~10–15 minutes (depends on network speed).  
**Output**: `nepali_indiccorp.jsonl` (~13 GB)

In [ ]:
out_path = RAW_DOWNLOAD_DIR / "nepali_indiccorp.jsonl"

# Target: enough docs to get 500M+ words (~17 words/doc avg -> ~30M docs)
LIMIT = 30_000_000

if out_path.exists() and out_path.stat().st_size > 1_000_000_000:  # >1GB means already downloaded
    print(f"Already exists: {out_path.name} ({out_path.stat().st_size / 1e9:.2f} GB). Skipping.")
    print("Delete the file to re-download.")
else:
    print(f"Loading IndicCorpV2 Nepali (npi_Deva) — streaming up to {LIMIT/1e6:.0f}M docs...")
    ds = load_dataset("ai4bharat/IndicCorpV2", "indiccorp_v2", split="npi_Deva", streaming=True)

    total_docs = 0
    total_words = 0

    with open(out_path, "w", encoding="utf-8") as f:
        for item in tqdm(ds, total=LIMIT, desc="Downloading IndicCorpV2 (Nepali)"):
            if total_docs >= LIMIT:
                break
            text = item["text"].strip()
            if text:
                f.write(json.dumps({"text": text, "source": "indiccorpv2_nepali"}, ensure_ascii=False) + "\n")
                total_docs += 1
                total_words += len(text.split())
            if total_docs % 5_000_000 == 0:
                print(f"  {total_docs/1e6:.0f}M docs, {total_words/1e6:.1f}M words, {out_path.stat().st_size/1e9:.2f} GB")

    print(f"\nDone. Saved {total_docs:,} docs ({total_words/1e6:.1f}M words) → {out_path.name}")
    print(f"File size: {out_path.stat().st_size / 1e9:.2f} GB")

## 3. Dataset Audit

Compute word and character counts for all downloaded JSONL files.

In [ ]:
import re

deva_re = re.compile(r"[\u0900-\u097F]")

all_files = sorted(RAW_DOWNLOAD_DIR.glob("*.jsonl"))
print(f"{'File':<40} {'Lines':>12} {'Words':>14} {'Chars':>16} {'Size (MB)':>10}")
print("-" * 96)
grand_lines = grand_words = grand_chars = 0

for fpath in all_files:
    lines = words = chars = 0
    with open(fpath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                text = json.loads(line).get("text", "")
            except Exception:
                text = line
            lines += 1
            words += len(text.split())
            chars += len(text)
    grand_lines += lines
    grand_words += words
    grand_chars += chars
    size_mb = fpath.stat().st_size / 1024 / 1024
    print(f"{fpath.name:<40} {lines:>12,} {words:>14,} {chars:>16,} {size_mb:>10,.1f}")

print("-" * 96)
print(f"{'TOTAL':<40} {grand_lines:>12,} {grand_words:>14,} {grand_chars:>16,}")
print(f"\nTotal words: {grand_words/1e6:.1f}M")

## 4. Sample Inspection

In [ ]:
import random

def inspect_samples(filepath, n=5):
    """Print random samples from a JSONL file."""
    if not filepath.exists():
        print(f"File not found: {filepath}")
        return
    # Read first 50K lines and sample from those
    samples = []
    with open(filepath, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 50000:
                break
            if line.strip():
                samples.append(line.strip())
    chosen = random.sample(samples, min(n, len(samples)))
    print(f"\n{'='*80}")
    print(f"Samples from: {filepath.name} ({len(samples):,} lines sampled)")
    print(f"{'='*80}")
    for i, line in enumerate(chosen, 1):
        try:
            text = json.loads(line).get("text", "")[:300]
        except:
            text = line[:300]
        print(f"\n[{i}] {text}")
        print("-" * 40)

inspect_samples(RAW_DOWNLOAD_DIR / "nepali_indiccorp.jsonl")
inspect_samples(RAW_DOWNLOAD_DIR / "nepali_wikipedia.jsonl")